# Apache Paimon Demo - Table Format for Data Lakes

## What is Apache Paimon?
- Open-source table format for data lakes
- Supports ACID transactions with primary keys
- Built on file formats (Parquet, ORC, Avro)
- Integrates with Spark, Flink, Trino, Hive

## This demo will demonstrate:
1. Creating Paimon tables with primary keys
2. UPSERT operations (update-or-insert)
3. Standard SQL queries on Paimon tables

## Setup: Import libraries and create Spark session

In [22]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

In [23]:
def create_spark_session():
    """Create Spark session configured for Paimon"""
    
    # Get JAR path
    base_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    paimon_jar = None
    
    jars_dir = os.path.join(base_dir, "jars")
    for file in os.listdir(jars_dir):
        if "paimon-spark" in file and file.endswith('.jar'):
            paimon_jar = os.path.join(jars_dir, file)
            break
    
    if not paimon_jar:
        raise FileNotFoundError("Paimon JAR not found. Please run setup.sh first.")
    
    warehouse_path = f"file://{base_dir}/warehouse/paimon"
    
    print(f"🔧 Configuring Spark with Paimon")
    print(f"   JAR: {os.path.basename(paimon_jar)}")
    print(f"   Warehouse: {warehouse_path}")
    
    spark = SparkSession.builder \
        .appName("Paimon Demo") \
        .config("spark.jars", paimon_jar) \
        .config("spark.sql.extensions", "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions") \
        .config("spark.sql.catalog.paimon", "org.apache.paimon.spark.SparkCatalog") \
        .config("spark.sql.catalog.paimon.warehouse", warehouse_path) \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("WARN")
    return spark

In [24]:
# Create the Spark session
spark = create_spark_session()
print("✅ Spark session with Paimon catalog created successfully!")

🔧 Configuring Spark with Paimon
   JAR: paimon-spark-3.4-1.3.0.jar
   Warehouse: file:///Users/yerachmielfeltzman/projects/personal/apache-paimon-demo/warehouse/paimon
✅ Spark session with Paimon catalog created successfully!


---
# DEMO 1: Creating Paimon Tables

## Key Concept: Paimon tables use primary keys for ACID operations

## Step 1: Create a database in the Paimon catalog

In [25]:
sql = "CREATE DATABASE IF NOT EXISTS paimon.demo"
print(f"SQL> {sql}")
spark.sql(sql)
print("✅ Database 'paimon.demo' created")
print("💡 Paimon stores data in the warehouse directory configured in Spark")

SQL> CREATE DATABASE IF NOT EXISTS paimon.demo
✅ Database 'paimon.demo' created
💡 Paimon stores data in the warehouse directory configured in Spark


## Step 2: Create a Paimon table with a PRIMARY KEY

**Key Concept**: Primary keys enable UPSERT operations (update-or-insert)

In [26]:
sql = """
    CREATE TABLE IF NOT EXISTS paimon.demo.employees (
        id BIGINT,
        name STRING,
        department STRING,
        salary INT,
        hire_date DATE
    ) TBLPROPERTIES (
        'primary-key' = 'id'
    )
"""
print(f"SQL> {sql.strip()}")
spark.sql(sql)
print("✅ Table 'employees' created with primary key on 'id' column")
print("💡 TBLPROPERTIES ('primary-key' = 'id') defines the primary key")

SQL> CREATE TABLE IF NOT EXISTS paimon.demo.employees (
        id BIGINT,
        name STRING,
        department STRING,
        salary INT,
        hire_date DATE
    ) TBLPROPERTIES (
        'primary-key' = 'id'
    )
✅ Table 'employees' created with primary key on 'id' column
💡 TBLPROPERTIES ('primary-key' = 'id') defines the primary key


## Step 3: Insert initial data into the Paimon table

In [27]:
base_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
employees_csv = os.path.join(base_dir, "data", "employees.csv")

df = spark.read.option("header", "true").option("inferSchema", "true").csv(employees_csv)
print("Loading from: data/employees.csv")
print("Schema inferred from CSV:")
df.printSchema()

Loading from: data/employees.csv
Schema inferred from CSV:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- hire_date: date (nullable = true)



In [28]:
sql = "INSERT INTO paimon.demo.employees"
print(f"SQL> {sql}  -- using DataFrame")
df.write.mode("overwrite").insertInto("paimon.demo.employees")
print("✅ Data loaded successfully")

SQL> INSERT INTO paimon.demo.employees  -- using DataFrame


✅ Data loaded successfully


## Step 4: Query the Paimon table

In [29]:
sql = "SELECT * FROM paimon.demo.employees ORDER BY id"
print(f"SQL> {sql}")
spark.sql(sql).show()
print("💡 Paimon tables are queried just like regular Spark tables")

SQL> SELECT * FROM paimon.demo.employees ORDER BY id
+---+-------------+-----------+------+----------+
| id|         name| department|salary| hire_date|
+---+-------------+-----------+------+----------+
|  1|Alice Johnson|Engineering| 95000|2023-01-15|
|  2|    Bob Smith|  Marketing| 75000|2023-02-20|
|  3|  Carol Davis|Engineering|105000|2023-03-10|
|  4| David Wilson|      Sales| 85000|2023-04-05|
|  5|    Eve Brown|Engineering| 98000|2023-05-12|
+---+-------------+-----------+------+----------+

💡 Paimon tables are queried just like regular Spark tables


---
# DEMO 2: UPSERT Operations (Primary Key in Action)

## Key Concept: Primary keys enable automatic UPSERT
- INSERT with existing key → UPDATE
- INSERT with new key → INSERT

## Step 1: View current data (before UPSERT)

In [30]:
sql = "SELECT id, name, salary FROM paimon.demo.employees ORDER BY id"
print(f"SQL> {sql}")
spark.sql(sql).show()
print("Note: Employee id=2 is 'Bob Smith' with salary 75000")

SQL> SELECT id, name, salary FROM paimon.demo.employees ORDER BY id
+---+-------------+------+
| id|         name|salary|
+---+-------------+------+
|  1|Alice Johnson| 95000|
|  2|    Bob Smith| 75000|
|  3|  Carol Davis|105000|
|  4| David Wilson| 85000|
|  5|    Eve Brown| 98000|
+---+-------------+------+

Note: Employee id=2 is 'Bob Smith' with salary 75000


## Step 2: Perform UPSERT operation

**Key Concept**: Inserting rows with id=2 (EXISTS) and id=6 (NEW)

In [31]:
sql = """
    INSERT INTO paimon.demo.employees VALUES
    (2, 'Bob Smith Jr.', 'Marketing', 80000, DATE '2023-02-20'),  -- Update existing
    (6, 'Frank Miller', 'Engineering', 110000, DATE '2023-06-01') -- Insert new
"""
print(f"SQL> {sql.strip()}")
spark.sql(sql)
print("✅ UPSERT completed")
print("💡 Notice: We used INSERT, but Paimon automatically:")
print("   • UPDATED the row where id=2 (Bob → Bob Smith Jr., 75k → 80k)")
print("   • INSERTED the new row where id=6 (Frank Miller)")

SQL> INSERT INTO paimon.demo.employees VALUES
    (2, 'Bob Smith Jr.', 'Marketing', 80000, DATE '2023-02-20'),  -- Update existing
    (6, 'Frank Miller', 'Engineering', 110000, DATE '2023-06-01') -- Insert new
✅ UPSERT completed
💡 Notice: We used INSERT, but Paimon automatically:
   • UPDATED the row where id=2 (Bob → Bob Smith Jr., 75k → 80k)
   • INSERTED the new row where id=6 (Frank Miller)


## Step 3: Verify the UPSERT result

In [32]:
sql = "SELECT id, name, salary FROM paimon.demo.employees ORDER BY id"
print(f"SQL> {sql}")
spark.sql(sql).show()
print("✅ Row id=2 was UPDATED, Row id=6 was INSERTED")

SQL> SELECT id, name, salary FROM paimon.demo.employees ORDER BY id
+---+-------------+------+
| id|         name|salary|
+---+-------------+------+
|  1|Alice Johnson| 95000|
|  2|Bob Smith Jr.| 80000|
|  3|  Carol Davis|105000|
|  4| David Wilson| 85000|
|  5|    Eve Brown| 98000|
|  6| Frank Miller|110000|
+---+-------------+------+

✅ Row id=2 was UPDATED, Row id=6 was INSERTED


## Step 4: Inspect table metadata

In [33]:
sql = "DESCRIBE EXTENDED paimon.demo.employees"
print(f"SQL> {sql}")
spark.sql(sql).select("col_name", "data_type").show(truncate=False)
print("💡 DESCRIBE EXTENDED shows table properties including primary key")

SQL> DESCRIBE EXTENDED paimon.demo.employees
+----------------------------+----------------------------------------------------------------------------------------------------------------------------+
|col_name                    |data_type                                                                                                                   |
+----------------------------+----------------------------------------------------------------------------------------------------------------------------+
|id                          |bigint                                                                                                                      |
|name                        |string                                                                                                                      |
|department                  |string                                                                                                                      |
|salary            

---
# DEMO 3: Querying Paimon Tables

## Key Concept: Paimon supports full SQL query capabilities

## Query 1: Aggregation - Department salary analysis

In [34]:
sql = """
    SELECT 
        department,
        COUNT(*) as employee_count,
        AVG(salary) as avg_salary,
        MAX(salary) as max_salary
    FROM paimon.demo.employees 
    GROUP BY department
    ORDER BY avg_salary DESC
"""
print(f"SQL> {sql.strip()}")
spark.sql(sql).show()

SQL> SELECT 
        department,
        COUNT(*) as employee_count,
        AVG(salary) as avg_salary,
        MAX(salary) as max_salary
    FROM paimon.demo.employees 
    GROUP BY department
    ORDER BY avg_salary DESC
+-----------+--------------+----------+----------+
| department|employee_count|avg_salary|max_salary|
+-----------+--------------+----------+----------+
|Engineering|             4|  102000.0|    110000|
|      Sales|             1|   85000.0|     85000|
|  Marketing|             1|   80000.0|     80000|
+-----------+--------------+----------+----------+



## Query 2: Filter - High-salary employees (>90k)

In [35]:
sql = """
    SELECT name, department, salary 
    FROM paimon.demo.employees 
    WHERE salary > 90000 
    ORDER BY salary DESC
"""
print(f"SQL> {sql.strip()}")
spark.sql(sql).show()

SQL> SELECT name, department, salary 
    FROM paimon.demo.employees 
    WHERE salary > 90000 
    ORDER BY salary DESC
+-------------+-----------+------+
|         name| department|salary|
+-------------+-----------+------+
| Frank Miller|Engineering|110000|
|  Carol Davis|Engineering|105000|
|    Eve Brown|Engineering| 98000|
|Alice Johnson|Engineering| 95000|
+-------------+-----------+------+



## Query 3: Date filter - Recent hires (2023)

In [36]:
sql = """
    SELECT name, department, hire_date 
    FROM paimon.demo.employees 
    WHERE year(hire_date) = 2023 
    ORDER BY hire_date
"""
print(f"SQL> {sql.strip()}")
spark.sql(sql).show()
print("💡 All standard Spark SQL functions work with Paimon tables")

SQL> SELECT name, department, hire_date 
    FROM paimon.demo.employees 
    WHERE year(hire_date) = 2023 
    ORDER BY hire_date
+-------------+-----------+----------+
|         name| department| hire_date|
+-------------+-----------+----------+
|Alice Johnson|Engineering|2023-01-15|
|Bob Smith Jr.|  Marketing|2023-02-20|
|  Carol Davis|Engineering|2023-03-10|
| David Wilson|      Sales|2023-04-05|
|    Eve Brown|Engineering|2023-05-12|
| Frank Miller|Engineering|2023-06-01|
+-------------+-----------+----------+

💡 All standard Spark SQL functions work with Paimon tables


---
# Key Takeaways

1. **Primary keys enable ACID UPSERT operations**
2. **INSERT automatically becomes UPDATE for existing keys**
3. **Standard Spark SQL works with Paimon tables**
4. **Paimon stores metadata alongside data files**

## Next Steps
- Run the cross-platform demo: `paimon-and-iceberg-cross-platform.ipynb`
- Explore Paimon's Iceberg compatibility feature

---
# Cleanup: Stop Spark session

In [37]:
spark.stop()
print("🛑 Spark session stopped")

🛑 Spark session stopped
